In [ ]:
from snowflake.snowpark.context     import get_active_session
from snowflake.snowpark.functions   import col,sum as sum_,avg, count
session = get_active_session()
print (session)

In [ ]:
df=session.table("TASTY_BYTES.RAW_POS.MENU")

In [ ]:
df.show()

In [ ]:
# .collect() pulls all rows into local memory as a list of Row objects
rows = df.collect()
print(f"Row count: {len(rows)}")
print(rows[0])

In [ ]:
from snowflake.snowpark.functions     import col
# filter the Dataframe to only include rows where truckbrand is "Freezing point"
df_freezing = df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
df_freezing.show()

In [ ]:
# select only specific columns 
df_selected = df_freezing.select(col("MENU_ITEM_NAME") ,col ("SALE_PRICE_USD"))
df_selected.show()

In [ ]:
df_result = (
        df
        .filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
        .select(col("MENU_ITEM_NAME") ,col ("SALE_PRICE_USD"))
)
df_result.show()

In [ ]:
df_sql = session.sql(""" SELECT MENU_ITEM_NAME,SALE_PRICE_USD FROM  TASTY_BYTES.RAW_POS.MENU WHERE  TRUCK_BRAND_NAME = 'Freezing Point' """)
df_sql.show()

In [ ]:
# create dataframe in python 
data = [("lemonade", 4.5),("dos", 6.5),("tres", 9.5)]
df_local = session.create_dataframe(data, schema=["item", "price"])
df_local.show()

# Lesson 2 Snowpark DataFrames - Part II

In [ ]:
df_sumary= (
    df
        .group_by(col("TRUCK_BRAND_NAME"))
        .agg(
            count(col("MENU_ITEM_NAME")).alias("item_count"),
            avg(col("SALE_PRICE_USD")).alias("avg_price"),
            sum_(col("SALE_PRICE_USD")).alias("sum_price")
        )
    .sort(col("avg_price").desc())
)
df_sumary.show()

In [ ]:
df.select(col("SALE_PRICE_USD"), col("COST_OF_GOODS_USD")).describe().show()

In [ ]:
df_pandas = df_sumary.to_pandas()
print(type(df_pandas))
df_pandas.head()

In [ ]:
session.sql(" CREATE SCHEMA IF NOT EXISTS TASTY_BYTES.ANALYTICS").collect()
df_sumary.write.mode("overwrite").save_as_table(
"TASTY_BYTES.ANALYTICS.brand_price_summary"
)

In [ ]:
session.table("TASTY_BYTES.ANALYTICS.brand_price_summary").show()